In [29]:
%load_ext autoreload
%autoreload 2

%autosave 120

import pandas as pd
import glob
import numpy as np
import os
import json
from tqdm import tqdm
from random import randrange
from modules.utilities import are_bugs_from_tangled, has_bug, is_same_history, parse_hash_delimited_string, has_one_bug

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Autosaving every 120 seconds


In [ ]:
df = pd.read_csv("./data/Complete_GoldSet.csv")
tnb = pd.read_csv("./data/GoldSet_TrueNotBuggyMethods.csv")

df = pd.merge(df, tnb, how="inner", on=["Project", "File", "CommitHash"])

In [18]:
row_to_remove = []
ages = []
for index, row in tqdm(df.iterrows()):
    file = row["File"]
    project = row["Project"]

    df_projct = pd.read_csv(f"./data/Processed/{project}.csv", delimiter="\t")
    age = df_projct.loc[df_projct["file"] == file]["Age"].values[0]
    
    if age < 365*2:
        ages.append(age)
        row_to_remove.append(index)

727it [01:04, 11.23it/s]


In [33]:
g_df = pd.read_csv("./data/Complete_GoldSet.csv")
g_df = g_df.drop(row_to_remove)

In [28]:
csv_files = glob.glob("./data/Processed/*.csv")

In [39]:
p_cnt = 1
true_not_buggy_methods = []
for csv_file in csv_files:
    df = pd.read_csv(csv_file,delimiter="\t")
    df = df.sample(frac=1).reset_index(drop=True)
    c_cnt = 0
    limit = 3 if p_cnt <= 26 else 2
    for index, row in df.iterrows():
        if row["Age"] < (365*2):
            continue

        if not has_bug(row["Buggycommiit"]) and is_same_history([row["RiskyCommit"], row["PotentiallyBuggycommit"],row["Buggycommiit"]]) and len(parse_hash_delimited_string(row["Buggycommiit"])) > 1:
            history = parse_hash_delimited_string(row["Buggycommiit"])
            history.reverse()
            
            project =  os.path.basename(csv_file).replace(".csv","")
            json_path = os.path.join("./data/source-methods/", project, row["file"])
            data = json.load(open(json_path))

            index = randrange(0, len(history) - 1)
            hash = data["changeHistory"][index]
            
            commit_data = data["changeHistoryDetails"][hash]
            change_data = commit_data["subchanges"][0] if "subchanges" in commit_data.keys() else commit_data
            if change_data["diff"] == "":
                continue
            
            exists = ((g_df['Project'] == project) & (g_df['File'] == row["file"])).any()
            if exists:
                continue

            true_not_buggy_methods.append(
                {
                    "Project": project,
                    "File": row["file"],
                    "CommitHash": hash,
                    "Diff": change_data["diff"],
                    "CommitMessage": change_data["commitMessage"],
                    "Decision": "NotBuggy"
                }
            )
            c_cnt = c_cnt + 1
        if c_cnt == limit:
            break
    p_cnt = p_cnt + 1            
true_not_buggy_gold_set = pd.DataFrame(true_not_buggy_methods)

In [41]:
true_not_buggy_gold_set

,Project,File,CommitHash,Diff,CommitMessage,Decision
0,spring-boot,3553.json,1e932860c46350bd8245559965a6922ec5ede8b2,"@@ -1,3 +1,3 @@\n-\tpublic Boolean getEnabled(...",Specify default micrometer values\n\nThis comm...,NotBuggy
1,spring-boot,5289.json,11d4426b4d7455f6eb780dbd566b7dfad2a39ef2,"@@ -1,9 +1,9 @@\n \tpublic RestTemplateBuilder...",Provide client factory with supplier in RestTe...,NotBuggy
2,spring-boot,2796.json,5ea3ab4595dffe60bb38b49126f32960cef2d4b8,"@@ -1,3 +1,3 @@\n-\t\tpublic Integer getFetchS...","Polish ""Allow to customize the JdbcTemplate""\n...",NotBuggy
3,guava,17767.json,6662777b4bbe4bb49f3d044281ea1929af38108d,"@@ -1,3 +1,3 @@\n- boolean isPartialView() {\...",Optimization of RegularImmutableMap.\n\n------...,NotBuggy
4,guava,11180.json,2da8a91a3702c6a318aaaeaa1f7164856a480d17,"@@ -1,5 +1,5 @@\n public static <A, B> Conve...",Use diamond operator in base+cache+concurrent....,NotBuggy
...,...,...,...,...,...,...
117,junit4,253.json,f1be7f250156ecc0e6b6c9d7005abb53359fe2fe,"@@ -1,6 +1,6 @@\n-\tprivate Matcher<?> createC...",Introduce withStacktrace() matcher\n\nStacktra...,NotBuggy
118,facebook-android-sdk,2369.json,019c5018c293e61e1b5150ef10192085f968d85e,"@@ -1,73 +1,81 @@\n private static boo...",Hash strings for codeless setup\n\nSummary: Ha...,NotBuggy
119,facebook-android-sdk,1054.json,19d1936c3b07d97d88646aeae30de747715e3248,"@@ -1,3 +1,4 @@\n public boolean canSh...",Facebook Android SDK 4.4\n,NotBuggy
120,titan,2682.json,1c91e6cd1ad51c838f3214ba899f6855d38b64ad,"@@ -1,3 +1,3 @@\n public void remove(Titan...",Continued interface refactoring\n,NotBuggy


In [42]:
g_df = pd.concat([g_df, true_not_buggy_gold_set])

In [43]:
g_df

,Project,File,CommitHash,Diff,Decision,CommitMessage,text
0,spring-boot,9232.json,ed15f742fd4eacc14b06908112ac4ca6ae4c0f90,"@@ -1,7 +1,7 @@\n \tpublic static String templ...",Buggy,Fix bug in GroovyTemplate convenience\n\nIt wa...,CommitMessage: Fix bug in GroovyTemplate conve...
2,spring-boot,510.json,99ae6dac5321a741d93ff5187fafb94c295a6928,"@@ -1,3 +1,3 @@\n-\t\tpublic CouchbaseEnvironm...",Buggy,Customize Couchbase's socket connect timeout\n...,CommitMessage: Customize Couchbase's socket co...
5,spring-boot,2051.json,4b4dc28a869e2f8b988f6ac6ea8a31c274477da5,"@@ -1,9 +1,10 @@\n \tprivate void logError(Ser...",Buggy,Support non-standard error codes with Abstract...,CommitMessage: Support non-standard error code...
9,guava,9470.json,2ee7f9da69308c56d5af71267e8b797cedaf31ba,"@@ -1,3 +1,5 @@\n public boolean hasEdgeConn...",Buggy,AbstractNetwork: fix bug in AbstractNetwork.ha...,CommitMessage: AbstractNetwork: fix bug in Abs...
10,guava,9471.json,2ee7f9da69308c56d5af71267e8b797cedaf31ba,"@@ -1,7 +1,7 @@\n public boolean hasEdgeConn...",Buggy,AbstractNetwork: fix bug in AbstractNetwork.ha...,CommitMessage: AbstractNetwork: fix bug in Abs...
...,...,...,...,...,...,...,...
117,junit4,253.json,f1be7f250156ecc0e6b6c9d7005abb53359fe2fe,"@@ -1,6 +1,6 @@\n-\tprivate Matcher<?> createC...",NotBuggy,Introduce withStacktrace() matcher\n\nStacktra...,NaN
118,facebook-android-sdk,2369.json,019c5018c293e61e1b5150ef10192085f968d85e,"@@ -1,73 +1,81 @@\n private static boo...",NotBuggy,Hash strings for codeless setup\n\nSummary: Ha...,NaN
119,facebook-android-sdk,1054.json,19d1936c3b07d97d88646aeae30de747715e3248,"@@ -1,3 +1,4 @@\n public boolean canSh...",NotBuggy,Facebook Android SDK 4.4\n,NaN
120,titan,2682.json,1c91e6cd1ad51c838f3214ba899f6855d38b64ad,"@@ -1,3 +1,3 @@\n public void remove(Titan...",NotBuggy,Continued interface refactoring\n,NaN
